### Implementation of k-means and k-means++ (Arthur & Vassilvitskii, 2007)
#### -standard k-means (Lloyd's algorithm) and k-means++ (D² seeding)

###### Dataset
###### Source: UCI Machine Learning Repository  
###### Dataset page: https://archive.ics.uci.edu/dataset/236/seeds - extract and save it locally

In [1]:
import numpy as np
import pandas as pd

np.random.seed(42) # set random seed across multiple runs

In [2]:
df = pd.read_csv("seeds_dataset.txt", sep=r"\s+", header=None) # read data from local text file
X = df.iloc[:, :7].values.astype(float) #consider first 7 columns
print("Seeds dataset shape:", X.shape)

Seeds dataset shape: (210, 7)


In [3]:
def euclidean_distance(a, b): # euclidean distance between 2 points
    return np.sqrt(np.sum((a - b)**2)) # result used to define the potential φ 

### K-means with random initialization (Lloyd’s algorithm)

In [4]:
def kmeans_random(X, k=3, max_iters=100, tol=1e-4):
    n, d = X.shape # n=number of points, d=number of features
    
    centers = X[np.random.choice(n, k, replace=False)].copy() # random k as centers

    for it in range(max_iters): # Lloyd iteration
        labels = np.array([
            np.argmin([euclidean_distance(x, c) for c in centers]) # assign closest center for x
            for x in X
        ])

        new_centers = np.array([
            X[labels == i].mean(axis=0) if np.any(labels == i) # new center for each cluster
            else centers[i] 
            for i in range(k)
        ])

        if np.linalg.norm(new_centers - centers) < tol:
            break # Stop if change is very small
        centers = new_centers
        
    phi = sum(euclidean_distance(X[i], centers[labels[i]])**2 
              for i in range(n)) # compute the k-means potential φ
    return centers, labels, phi, it + 1

### K-means++ with D2 seeding
#### Algorithm from the paper - k-means++: The Advantages of Careful Seeding
##### 1a.Choose an initial center c1 uniformly at random from X.
##### 1b.Choose the next center ci, selecting ci= x′ ∈ X with probability D(x′)2/x∈XD(x)2
##### 1c.Repeat Step1b until we have chosen a total of k centers.
##### 2-4. Proceed as with the standard k-means algorithm.
##### We call the weighting used in Step 1b simply “D2 weighting”.

In [5]:
def kmeans_pp(X, k=3, max_iters=100, tol=1e-4):
    n, d = X.shape
    centers = [X[np.random.randint(n)]]  # Step 1a: pick first center randomly

    while len(centers) < k: # Steps 1b/1c: choose remaining centres with D^2 weighting
        dist_sq = np.array([
            min(euclidean_distance(x, c)**2 for c in centers)
            for x in X
        ])
        probs = dist_sq / dist_sq.sum() # k-means++ seeding as mentioned in the paper
        centers.append(X[np.random.choice(n, p=probs)])
    centers = np.array(centers)

    for it in range(max_iters): # 2-4 After seeding, run the same Lloyd iterations
        labels = np.array([
            np.argmin([euclidean_distance(x, c) for c in centers])
            for x in X
        ])
        new_centers = np.array([
            X[labels == i].mean(axis=0) if np.any(labels == i) else centers[i]
            for i in range(k)
        ])
        if np.linalg.norm(new_centers - centers) < tol:
            break # Stop if change is very small
        centers = new_centers

    phi = sum(euclidean_distance(X[i], centers[labels[i]])**2 
              for i in range(n)) # compute the final potential φ
    return centers, labels, phi, it + 1

### Run both methods several times 
#### calculate the number of iterations and then compute averages and minimum values

In [6]:
def run_experiments(X, k=3, runs=50): #list stores result for each run
    phi_random_list, it_random_list = [], []
    phi_pp_list, it_pp_list = [], []

    for _ in range(runs):
        _, _, phi_r, it_r = kmeans_random(X, k) # k-means with random centers
        _, _, phi_p, it_p = kmeans_pp(X, k) # k-means++ with D2 seeding
        
# collects the result
        phi_random_list.append(phi_r)
        it_random_list.append(it_r)
        phi_pp_list.append(phi_p)
        it_pp_list.append(it_p)

    results = {
        "phi_random_avg": np.mean(phi_random_list),
        "phi_random_min": np.min(phi_random_list),
        "phi_pp_avg": np.mean(phi_pp_list),
        "phi_pp_min": np.min(phi_pp_list),
        "it_random_avg": np.mean(it_random_list),
        "it_pp_avg": np.mean(it_pp_list),
    }
    return results

In [7]:
results = run_experiments(X, k=3, runs=50) # run both multiple times

df_results = pd.DataFrame({
    "method": ["k-means (random)", "k-means++"],
    "avg_phi": [results["phi_random_avg"], results["phi_pp_avg"]],
    "min_phi": [results["phi_random_min"], results["phi_pp_min"]],
    "avg_iterations": [results["it_random_avg"], results["it_pp_avg"]],
})
df_results

,method,avg_phi,min_phi,avg_iterations
0,k-means (random),587.848155,587.318612,8.44
1,k-means++,588.108837,587.318612,7.30


#### k-Hyperparameter Testing

In [8]:
def evaluate_k_values(X, k_values=(2, 3, 4, 5), runs=30, max_iters=100, tol=1e-4):
    rows = []

    for k in k_values: # test each k value and store it in the list
        phi_r_list, it_r_list = [], []
        phi_p_list, it_p_list = [], []

        for _ in range(runs):
            _, _, phi_r, it_r = kmeans_random(X, k=k, max_iters=max_iters, tol=tol) # k-means
            _, _, phi_p, it_p = kmeans_pp(X, k=k, max_iters=max_iters, tol=tol) # k-means++
# collects the result
            phi_r_list.append(phi_r)
            it_r_list.append(it_r)
            phi_p_list.append(phi_p)
            it_p_list.append(it_p)

        rows.append({
            "k": k,
            "method": "k-means (random)",
            "avg_phi": np.mean(phi_r_list),
            "min_phi": np.min(phi_r_list),
            "avg_iterations": np.mean(it_r_list),
        })
        rows.append({
            "k": k,
            "method": "k-means++",
            "avg_phi": np.mean(phi_p_list),
            "min_phi": np.min(phi_p_list),
            "avg_iterations": np.mean(it_p_list),
        })

    return pd.DataFrame(rows) # results table

In [9]:
df_k_eval = evaluate_k_values(X, k_values=(2, 3, 4, 5), runs=30) # table comparing k-means and k-means++ for k=2,3,4,5
df_k_eval

,k,method,avg_phi,min_phi,avg_iterations
0,2,k-means (random),1011.755090,1011.712345,6.466667
1,2,k-means++,1011.774735,1011.612265,5.233333
2,3,k-means (random),587.855184,587.318612,9.533333
3,3,k-means++,587.859728,587.318612,6.866667
4,4,k-means (random),480.290963,471.003396,8.933333
5,4,k-means++,481.760110,471.027146,7.800000
6,5,k-means (random),412.899326,385.507292,10.233333
7,5,k-means++,401.218153,385.507292,9.500000


# Overview of tools used

### Perplexity AI
###### AI tools were used for literature study, to support understanding of the mathematical formulas and algorithmic steps involved in the k-means and k-means++ algorithms.
###### The implementation, experiments, and analysis presented here are my own work.

### Source of Dataset - UCI ML Repository
